# What does the client's scoring rule do with a policy?

This notebook loads the bookings, adds the two columns the scoring rule needs, and scores the simplest policy there is: sell nothing extra. It runs in under a minute. Change nothing here; your team's work goes in `src/`.

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np

import contract
import data

In [ ]:
bookings = data.read_raw()
bookings["arrival_date"] = data.arrival_date(bookings)
bookings["nights"] = bookings["stays_in_weekend_nights"] + bookings["stays_in_week_nights"]
print(f"{len(bookings):,} bookings, arrivals {bookings.arrival_date.min().date()} to {bookings.arrival_date.max().date()}")

119,390 bookings, arrivals 2015-07-01 to 2017-08-31


In [ ]:
train, choose, proof = contract.split_periods(bookings)
for name, part in [("train", train), ("choose", choose), ("proof", proof)]:
    print(f"{name:>6}: {len(part):>7,} bookings")

 train:  49,157 bookings
choose:  29,546 bookings
 proof:  40,687 bookings


## What does selling nothing extra cost on the choose period?

Every cancellation leaves a room empty and nobody is walked. This is a floor, not the client's rule: the client already oversells with a flat rate, and your savings are measured against that.

In [ ]:
nights = contract.nightly_table(choose, prob=np.zeros(len(choose)))
nights.head()

book  cancelled  expected  spread  stay_value
hotel      arrival_date                                               
City Hotel 2016-07-01     141         68       0.0     0.0  319.747589
           2016-07-02      80         33       0.0     0.0  397.106875
           2016-07-03      85         21       0.0     0.0  478.769529
           2016-07-04      92         30       0.0     0.0  440.365543
           2016-07-05      74         14       0.0     0.0  405.705811

In [ ]:
contract.night_cost(nights, oversell=np.zeros(len(nights)))

{'cost_eur': 4155340.86,
 'empty_rooms': 10894,
 'walked_guests': 0,
 'walks_per_1000': 0.0,
 'ceiling_met': True,
 'nights': 368}

## What comes next?

- Check every number in `BRIEF.md` against `bookings`, and record each one in the charter.
- Formalise the client's flat rule in `src/backtest.py` and score it on these same nights.
- Leave the proof period alone until every setting is frozen.

## Charter draft context and number checks

This table mirrors Charter question 6. ‘Not checked’ means the current charter draft still needs a deliberate check and reconciliation. The notebook already contains cached outputs from an earlier run: `00_start_here` shows 119,390 bookings, arrivals from 2015-07-01 through 2017-08-31, and train/choose/proof row counts; `data_explore` shows the column index. Those cached outputs have not been rerun in this task.

The charter’s booking-level decision-unit guess is intentionally weak and probably wrong. The scoring contract defines the action unit as one hotel on one arrival night.

| Brief item | Status in current charter | Existing notebook evidence / next check |
|---|---|---|
| About one booking in four is cancelled | Not checked from CSV | Compute cancellation share overall and by hotel/period.
| Flat cancellation-rate rule and ‘a few rooms’ safety margin | Rate and margin not checked | Ask the revenue manager for the current margin; formalize baseline.
| One guest in 100 / 10 walks per 1,000 arrivals | 1% arithmetic match; observed rate not checked | Confirm denominator; use the explicit 10/1,000 contract ceiling.
| EUR 300 per walked guest | Brief-stated assumption | Not verifiable from bookings; use as the scoring cost unless client changes it.
| Train before 1 July 2016 | Period stated; boundary not independently checked | Cached split counts: 49,157 train bookings. Inspect arrival-date boundary.
| Choose July–December 2016 | Period stated; boundary not independently checked | Cached split counts: 29,546 choose bookings. Inspect arrival-date boundary.
| Proof January–August 2017 | Period stated; proof not independently checked | Cached split counts: 40,687 proof bookings. Hold back and run the proof once.
| Next fourteen nights | Output requirement | MVP has not been run.
| 119,390 bookings / 32 columns | Charter says not checked yet | Existing caches show 119,390 bookings and the column index; rerun and confirm count is 32.
| Vendor demo claimed 95% accuracy | Unverified | No demo evidence supplied; do not use as an acceptance target.

These cached values are inherited notebook outputs, not results computed during this request. Update the charter only after checking the source data and confirming the outputs.

In [ ]:
import pandas as pd

draft_checks = [
    {"Brief item": "About one booking in four is cancelled", "Charter status": "Not checked from CSV", "Next check": "Compute overall and by hotel/period"},
    {"Brief item": "Flat rule and safety margin", "Charter status": "Current rate/margin not checked", "Next check": "Ask revenue manager for current margin"},
    {"Brief item": "1 walk per 100 / 10 per 1,000", "Charter status": "Equivalent 1% arithmetic; observed rate not checked", "Next check": "Confirm denominator; use contract ceiling"},
    {"Brief item": "EUR 300 per walked guest", "Charter status": "Brief-stated assumption", "Next check": "Use as scoring cost unless client changes it"},
    {"Brief item": "Train / choose / proof periods", "Charter status": "Dates stated; boundaries not independently checked", "Next check": "Inspect arrival-date boundaries"},
    {"Brief item": "Fourteen-night output", "Charter status": "Requirement; not run", "Next check": "Build and run the MVP"},
    {"Brief item": "119,390 bookings / 32 columns", "Charter status": "Not checked yet", "Next check": "Rerun count and confirm 32 columns"},
    {"Brief item": "Vendor demo 95% accuracy", "Charter status": "Unverified", "Next check": "Request evidence; use business costs for acceptance"},
]
pd.DataFrame(draft_checks)